# ChartSense — Data Exploration

Exploratory analysis of the ChartSense dataset: synthetic chart images, pattern labels, and direction annotations.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

DATA_DIR = Path('../data')
SYNTHETIC_DIR = DATA_DIR / 'synthetic'
BG_COLOR = '#0f1117'
TEXT_COLOR = '#e0e0e0'

df = pd.read_csv(DATA_DIR / 'labels.csv')

print('Shape:', df.shape)
df.head()

## 2. Class Distribution

In [ ]:
counts = df['pattern'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

bars = ax.bar(counts.index, counts.values, color='#2ecc71', edgecolor='#27ae60', linewidth=0.8)

for bar, val in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        str(val),
        ha='center', va='bottom',
        color=TEXT_COLOR, fontsize=10
    )

ax.set_title('Pattern Class Distribution', color=TEXT_COLOR, fontsize=14, pad=14)
ax.set_xlabel('Pattern Class', color=TEXT_COLOR, fontsize=11)
ax.set_ylabel('Count', color=TEXT_COLOR, fontsize=11)
ax.tick_params(colors=TEXT_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor('#333344')
ax.set_xticklabels(counts.index, rotation=25, ha='right', color=TEXT_COLOR)
ax.yaxis.set_tick_params(labelcolor=TEXT_COLOR)

plt.tight_layout()
plt.show()

## 3. Sample Charts

In [ ]:
classes = [
    'bull_flag',
    'descending_triangle',
    'double_bottom',
    'double_top',
    'head_and_shoulders',
    'random',
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.patch.set_facecolor(BG_COLOR)
fig.suptitle('Sample Chart per Pattern Class', color=TEXT_COLOR, fontsize=14, y=1.01)

for ax, cls in zip(axes.flat, classes):
    row = df[df['pattern'] == cls].iloc[0]
    img_path = SYNTHETIC_DIR / row['filename']
    img = Image.open(img_path)

    ax.imshow(img)
    ax.set_title(cls.replace('_', ' ').title(), color=TEXT_COLOR, fontsize=10, pad=6)
    ax.set_facecolor(BG_COLOR)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Label Analysis

In [ ]:
dir_counts = df['direction'].value_counts()
print('Direction value counts:')
print(dir_counts.to_string())

PALETTE = ['#2ecc71', '#e74c3c', '#95a5a6']

fig, ax = plt.subplots(figsize=(6, 6))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

wedges, texts, autotexts = ax.pie(
    dir_counts.values,
    labels=dir_counts.index,
    autopct='%1.1f%%',
    colors=PALETTE[:len(dir_counts)],
    startangle=140,
    wedgeprops=dict(edgecolor=BG_COLOR, linewidth=2),
)

for t in texts:
    t.set_color(TEXT_COLOR)
    t.set_fontsize(11)
for at in autotexts:
    at.set_color(BG_COLOR)
    at.set_fontsize(10)
    at.set_fontweight('bold')

ax.set_title('Direction Label Distribution', color=TEXT_COLOR, fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 5. Training Curves

Training loss and accuracy curves are printed to the console during training and are not persisted to disk.

To run training and observe live metrics:

```bash
python -m chartsense.training.train
```

The trainer logs per-epoch metrics in the format:

```
Epoch 01/20 | train_loss: 1.432 | val_loss: 1.198 | val_acc: 0.621
```

**Phase 1** trains the full backbone + classification head on synthetic data.  
**Phase 2** fine-tunes only the direction head on real yfinance charts (`data/real/`).

## 6. Key Findings

### Dataset
- **3,000 synthetic charts** across **6 pattern classes** — balanced at ~500 samples per class
- Classes: `bull_flag`, `descending_triangle`, `double_bottom`, `double_top`, `head_and_shoulders`, `random`
- Direction labels: bullish / bearish / neutral

### Pattern Classification (Phase 1 — Synthetic)
- **Overall F1: 0.94**
- Weakest class: `random` at **F1 = 0.80** — the model occasionally assigns structure to structureless noise, mirroring the *apophenia bias* observed in human traders who perceive patterns in random price action

### Direction Prediction
| Evaluation set | Accuracy |
|---|---|
| Synthetic held-out | **97%** |
| Real yfinance charts | **41.7%** |

The large synthetic → real gap confirms that synthetic data alone is insufficient for direction prediction in live markets.

### Real-data Fine-tuning (Phase 2)
- **300 charts** fetched via `yfinance` from **15 stocks**
- Only the direction head weights are updated; the backbone is frozen
- Fine-tuning raises real-data direction accuracy from near-chance to 41.7%

### Notable Observation
The `random` class confusion is not merely a model limitation — it reflects a well-documented cognitive bias. Human traders consistently over-detect chart patterns in random walks, and the model has learned a similar heuristic from the data distribution.